# Figure 2 v2: station observations

Memplot ulang Figure 2 dengan titik stasiun BMKG di panel (c) dan garis running correlation hasil rerata hujan stasiun di panel (d).


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

os.environ.setdefault('MPLCONFIGDIR', '/tmp/fig2-mpl')
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
                     'font.size': 24, 'axes.labelsize': 24, 'axes.titlesize': 24,
                     'xtick.labelsize': 24, 'ytick.labelsize': 24, 'figure.dpi': 300})
FIG2_DIR = Path.cwd().resolve()
DATA_DIR = FIG2_DIR / 'data'
SPATIAL_CACHE = DATA_DIR / 'rainfall_corr_cache_1981_2025.nc'
RUNNING_CORR_CACHE = DATA_DIR / 'gambarL.2_running_corr_box1.csv'
STATION_CORR_CACHE = DATA_DIR / 'running_corr_stasiun_box1.csv'
STATION_META = DATA_DIR / 'stasiun_box1.csv'
OUTPUT_PATH = FIG2_DIR / 'Fig2_v2.png'
for path in [SPATIAL_CACHE, RUNNING_CORR_CACHE, STATION_CORR_CACHE, STATION_META]:
    assert path.exists(), f'Missing {path}; run running_corr_sta.ipynb first.' 


## Load cached maps, existing lines, and station output


In [ ]:
rain_ds = xr.open_dataset(SPATIAL_CACHE)
running_corr_df = pd.read_csv(RUNNING_CORR_CACHE)
station_corr_df = pd.read_csv(STATION_CORR_CACHE)
stations = pd.read_csv(STATION_META)

rain_corr_p1 = rain_ds['rain_corr_past']
rain_corr_p2 = rain_ds['rain_corr_recent']
rain_corr_diff = rain_ds['rain_corr_recent_minus_past']
rain_corr_p1_sig = rain_ds['rain_corr_past_sig']
rain_corr_p2_sig = rain_ds['rain_corr_recent_sig']
rain_corr_diff_sig = rain_ds['rain_corr_recent_minus_past_sig']
print(f'Plotting {len(stations)} stations and {station_corr_df.running_corr.notna().sum()} valid station running-correlation values.')


## Create Figure 2 v2


In [ ]:
corr_levels = np.arange(-1, 1.01, 0.05)
corr_ticks = np.arange(-1, 1.01, 0.25)
mc_extent, mc_xticks, mc_yticks = [80, 160, -20, 20], np.arange(90, 161, 20), np.arange(-20, 21, 10)
vimfc_domain = [95, 125, -6, 2]
dataset_order = ['MSWEP', 'ERA5', 'CHIRPS', 'GPCC', 'SAOBS', 'GPCP']
dataset_colors = {'MSWEP':'black','ERA5':'#17b978','CHIRPS':'#ff5e62','GPCC':'#ff9900','SAOBS':'#a06ee1','GPCP':'#085f56'}

fig = plt.figure(figsize=(19, 15))
gs = fig.add_gridspec(2, 4, hspace=0.1, wspace=0.35)
ax_p1 = fig.add_subplot(gs[0, :2], projection=ccrs.PlateCarree(central_longitude=180))
ax_p2 = fig.add_subplot(gs[0, 2:], projection=ccrs.PlateCarree(central_longitude=180))
ax_diff = fig.add_subplot(gs[1, :2], projection=ccrs.PlateCarree(central_longitude=180))
ax_running = fig.add_subplot(gs[1, 2:])
map_panels = [(ax_p1,rain_corr_p1,rain_corr_p1_sig,'(a) Rainfall correlation: P1','bwr','neither'),
              (ax_p2,rain_corr_p2,rain_corr_p2_sig,'(b) Rainfall correlation: P2','bwr','neither'),
              (ax_diff,rain_corr_diff,rain_corr_diff_sig,'(c) Rainfall correlation difference: P2 − P1','BrBG','both')]
top_img = diff_img = None
for ax, data, sig_mask, title, cmap, extend in map_panels:
    img = data.reset_coords(drop=True).plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(), cmap=cmap, levels=corr_levels, extend=extend, add_colorbar=False, add_labels=False, infer_intervals=True)
    if ax is ax_diff: diff_img = img
    else: top_img = img
    sig_plot = (sig_mask.fillna(False).astype(np.int8).coarsen(lat=8, lon=8, boundary='trim').max() > 0)
    yy, xx = np.where(sig_plot.values)
    ax.scatter(sig_plot.lon.values[xx], sig_plot.lat.values[yy], s=8, c='k', marker='.', linewidths=0, alpha=.55, transform=ccrs.PlateCarree(), zorder=4)
    ax.coastlines(resolution='50m', linewidth=.6, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=.3, color='gray', alpha=.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree()); ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter(cardinal_labels={'east':'E','west':'W'})); ax.yaxis.set_major_formatter(LatitudeFormatter(cardinal_labels={'north':'N','south':'S'}))
    ax.tick_params(direction='out', top=True, right=True, labelsize=24); ax.set_title(title, loc='left', fontweight='bold', pad=12)

ax_diff.add_patch(Rectangle((vimfc_domain[0],vimfc_domain[2]), vimfc_domain[1]-vimfc_domain[0], vimfc_domain[3]-vimfc_domain[2], fill=False, edgecolor='black', linewidth=2.8, transform=ccrs.PlateCarree(), zorder=6))
ax_diff.scatter(stations.longitude, stations.latitude, s=38, marker='o', facecolor='#56B4E9', edgecolor='black', linewidth=.7, transform=ccrs.PlateCarree(), zorder=7, label=f'Stations (n={len(stations)})')
ax_diff.legend(loc='lower left', fontsize=13, frameon=True, facecolor='white', edgecolor='grey', framealpha=.95)

for name in dataset_order:
    series = running_corr_df.query('window == 15 and dataset == @name')
    ax_running.plot(series.djf_year, series.running_corr, linewidth=3.5 if name=='MSWEP' else 2, color=dataset_colors[name], label=name)
ax_running.plot(station_corr_df.djf_year, station_corr_df.running_corr, color='black', linewidth=3.5, linestyle='--', label='Station', zorder=8)
ax_running.axhline(0, color='grey', linewidth=1, linestyle='--')
ax_running.set(ylim=(-1,1), xlim=(1981,2025), yticks=[-1,-.5,0,.5,1], xticks=np.arange(1985,2021,5))
ax_running.grid(True, linestyle='--', linewidth=.5, alpha=.4); ax_running.set_xlabel('Year', labelpad=18); ax_running.set_ylabel(r'$\it{r}$', labelpad=1)
ax_running.yaxis.tick_right(); ax_running.yaxis.set_label_position('right'); ax_running.set_title('d) Running Correlation 15 years', loc='left', pad=15)
legend_handles, legend_labels = ax_running.get_legend_handles_labels()
legend_by_label = dict(zip(legend_labels, legend_handles))
legend_order = ['MSWEP', 'Station', 'ERA5', 'CHIRPS', 'GPCC', 'SAOBS', 'GPCP']
ax_running.tick_params(labelsize=24); ax_running.legend([legend_by_label[x] for x in legend_order], legend_order, ncol=3, frameon=True, loc='upper left', fontsize=14, facecolor='white', edgecolor='grey', framealpha=.95, columnspacing=.9, handlelength=2.5)

fig.subplots_adjust(left=.06, right=.97, bottom=.10, top=.94)
fig.canvas.draw(); pos_diff, pos_running = ax_diff.get_position(), ax_running.get_position(); ax_running.set_position([pos_running.x0,pos_diff.y0,pos_diff.width,pos_diff.height])
pos_p1, pos_p2, pos_diff = ax_p1.get_position(), ax_p2.get_position(), ax_diff.get_position()
top_cax=fig.add_axes([.25,pos_p1.y0-.070,.50,.022]); top_cbar=fig.colorbar(top_img,cax=top_cax,orientation='horizontal',ticks=corr_ticks); top_cbar.set_label(r'$\it{r}$',rotation=0,labelpad=14,fontsize=24); top_cbar.ax.tick_params(labelsize=24); top_cbar.ax.yaxis.set_label_position('right')
diff_cax=fig.add_axes([pos_diff.x0,pos_diff.y0-.067,pos_diff.width,.022]); diff_cbar=fig.colorbar(diff_img,cax=diff_cax,orientation='horizontal',ticks=corr_ticks,extend='both'); diff_cbar.set_label(r'$\Delta r$',rotation=0,labelpad=18,fontsize=24); diff_cbar.ax.tick_params(labelsize=24); diff_cbar.ax.yaxis.set_label_position('right')
fig.savefig(OUTPUT_PATH,dpi=300,bbox_inches='tight')
print(f'Saved: {OUTPUT_PATH}')
plt.show()
rain_ds.close()


Panel (c) menampilkan posisi semua stasiun yang dipakai dalam rerata hujan. Garis hitam pada panel (d) adalah hasil dari `running_corr_stasiun_box1.csv`.
